# Initial-margin CVaR MKABSDE

This notebook displays the retained quadrature campaign, both proxy BSDEs, and the common-path comparison of three conditional-CVaR estimators.

## Complete reproduction commands

```bash
python -m initial_margin.run_campaign \
  --product all --steps 50 100 200 \
  --seeds 42 43 44 45 46 47 48 49 50 51 \
  --paths 200000 --degree 2 --device cuda

python -m initial_margin.compare_risk_estimators \
  --product all --steps 200 --seeds 42 43 44 \
  --paths 200000 --degree 2 --device cuda

python -m initial_margin.deterministic_reference \
  --product all --steps 50 100 200 --space-steps 6000 \
  --expectation-nodes 24 \
  --output initial_margin/results/deterministic_reference.json
```

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
sys.path.insert(0, str(root))
from initial_margin.evaluation import ARTICLE_REFERENCE_VALUES

result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')
reference_payload = json.loads((result_root / 'deterministic_reference.json').read_text())
direct_references = {(row['product'], row['time_steps']): row for row in reference_payload['rows']}
method_payload = json.loads((result_root / 'cvar_method_comparison.json').read_text())

## Relative margin valuation adjustment

Every number is $100\,\mathrm{MVA}_0/V_0^{bs}=100\,(V_0-V_0^{bs})/V_0^{bs}$, expressed in percent. The raw MVA and total price remain in the JSON artifacts; this is not the CVaR collateral.

In [2]:
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    base_price = payload['rows'][0]['metrics']['black_scholes_no_im']['price']
    article = ARTICLE_REFERENCE_VALUES[product]
    degree = payload['summary'][0]['hermite_degree']
    lines = [
        f'### ATM {product} — Hermite degree {degree}',
        '',
        '| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |',
        '|---:|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {row['direct_mkabsde_mva_percent_of_black_scholes_mean']:.6f} ± {row['direct_mkabsde_mva_percent_of_black_scholes_std']:.2e} | "
            f"{direct_references[(product, row['time_steps'])]['mva_percent_of_black_scholes']:.6f} | "
            f"{row['nonlinear_proxy_mva_percent_of_black_scholes_mean']:.6f} ± {row['nonlinear_proxy_mva_percent_of_black_scholes_std']:.2e} | "
            f"{row['linear_proxy_mva_percent_of_black_scholes_mean']:.6f} ± {row['linear_proxy_mva_percent_of_black_scholes_std']:.2e} |"
        )
    lines.append(
        f"| Reference value from Agarwal et al. | — | — | {100 * (article['nonlinear_proxy_price'] - base_price) / base_price:.6f} | {100 * (article['linear_proxy_price'] - base_price) / base_price:.6f} |"
    )
    display(Markdown('\n'.join(lines)))

### ATM call — Hermite degree 2

| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 0.861525 ± 1.31e-03 | 0.861609 | 1.010607 ± 1.32e-03 | 1.007173 ± 1.32e-03 |
| 100 | 0.859277 ± 9.07e-04 | 0.859533 | 1.007756 ± 9.32e-04 | 1.004309 ± 9.30e-04 |
| 200 | 0.858389 ± 1.10e-03 | 0.858281 | 1.006538 ± 1.14e-03 | 1.003083 ± 1.14e-03 |
| Reference value from Agarwal et al. | — | — | 1.002170 | 1.057365 |

### ATM butterfly — Hermite degree 2

| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 0.764252 ± 7.51e-04 | 0.766698 | 0.724722 ± 6.56e-04 | 0.723675 ± 6.56e-04 |
| 100 | 0.771739 ± 6.19e-04 | 0.773731 | 0.732213 ± 5.26e-04 | 0.731134 ± 5.27e-04 |
| 200 | 0.774380 ± 4.31e-04 | 0.776302 | 0.735188 ± 2.07e-04 | 0.734097 ± 2.07e-04 |
| Reference value from Agarwal et al. | — | — | 0.700663 | 0.700663 |

## Conditional-CVaR estimator audit

Only the CVaR engine changes below. The outer paths, Hermite regressions, and direct MKABSDE update are identical.

In [3]:
lines = [
    '| Product | CVaR estimator | $100\,\mathrm{MVA}_0/V_0^{bs}$ (%) | GPU time | price difference vs quadrature |',
    '|---|---|---:|---:|---:|',
]
for row in method_payload['summary']:
    lines.append(
        f"| {row['product']} | {row['method']} | {row['mva_percent_of_black_scholes_mean']:.6f} ± "
        f"{row['mva_percent_of_black_scholes_std']:.2e} | {row['gpu_seconds_mean']:.2f} s | "
        f"{row['price_difference_vs_quadrature_mean']:+.2e} |"
    )
display(Markdown('\n'.join(lines)))

| Product | CVaR estimator | $100\,\mathrm{MVA}_0/V_0^{bs}$ (%) | GPU time | price difference vs quadrature |
|---|---|---:|---:|---:|
| butterfly | quadrature | 0.774071 ± 2.60e-04 | 14.90 s | +0.00e+00 |
| butterfly | nested_mc | 0.777692 ± 7.80e-03 | 60.10 s | +1.12e-05 |
| butterfly | learning | 0.785762 ± 4.32e-03 | 51.94 s | +3.61e-05 |
| call | quadrature | 0.858553 ± 1.49e-03 | 5.22 s | +0.00e+00 |
| call | nested_mc | 0.859383 ± 1.21e-03 | 43.63 s | +1.80e-05 |
| call | learning | 0.860903 ± 2.90e-03 | 51.16 s | +5.11e-05 |

## Interpretation

The three risk engines agree on the direct time-zero price to better than $10^{-4}$, so the direct/proxy gap is not caused by Gauss--Legendre quadrature. Quadrature is retained because it is deterministic conditional on the current regression and is the fastest method in dimension one. The deterministic state-space calculation is a separate same-grid regression check; it is not an exact continuous-time solution.